In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

In [ ]:
deliveries_train = pd.read_csv("deliveries_train.csv")
deliveries_test = pd.read_csv("deliveries_test.csv")

print(deliveries_train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               4000 non-null   object 
 1   order_time             4000 non-null   object 
 2   main_dish              4000 non-null   object 
 3   order_price            4000 non-null   float64
 4   restaurant_lat         4000 non-null   float64
 5   restaurant_lng         4000 non-null   float64
 6   customer_lat           4000 non-null   float64
 7   customer_lng           4000 non-null   float64
 8   delivery_neighborhood  4000 non-null   object 
 9   weather                4000 non-null   object 
 10  traffic_level          4000 non-null   object 
 11  restaurant_busy_level  4000 non-null   object 
 12  courier_name           4000 non-null   object 
 13  courier_vehicle        4000 non-null   object 
 14  customer_first_name    4000 non-null   object 
 15  app_

In [ ]:
!pip install haversine

In [ ]:
from haversine import haversine

deliveries_train["distance"] = deliveries_train.apply(
    lambda row: haversine(
        (row["restaurant_lat"], row["restaurant_lng"]),
        (row["customer_lat"], row["customer_lng"])
    ),
    axis=1
)

deliveries_test["distance"] = deliveries_test.apply(
    lambda row: haversine(
        (row["restaurant_lat"], row["restaurant_lng"]),
        (row["customer_lat"], row["customer_lng"])
    ),
    axis=1
)



In [ ]:
le_int = LabelEncoder()
le_veh = LabelEncoder()
le_we = LabelEncoder()

le_int.classes_ = np.array(["low", "medium", "high"])
le_veh.classes_ = np.array(["walking", "bike", "motorcycle", "car"])
le_we.classes_ = np.array(["clear", "cloudy", "rain", "storm"])

int_cols = ["traffic_level", "restaurant_busy_level"]

for col in int_cols:
  deliveries_train[col] = le_int.transform(deliveries_train[col])
  deliveries_test[col] = le_int.transform(deliveries_test[col])

In [ ]:
deliveries_train["courier_vehicle"] = le_veh.transform(deliveries_train["courier_vehicle"])
deliveries_test["courier_vehicle"] = le_veh.transform(deliveries_test["courier_vehicle"])
deliveries_train["weather"] = le_we.transform(deliveries_train["weather"])
deliveries_test["weather"] = le_we.transform(deliveries_test["weather"])

In [37]:
deliveries_train

,order_id,order_time,main_dish,order_price,restaurant_lat,restaurant_lng,customer_lat,customer_lng,delivery_neighborhood,weather,traffic_level,restaurant_busy_level,courier_name,courier_vehicle,customer_first_name,app_version,delivery_time_minutes,distance
0,DT04768,8,poke_bowl,47.49,-23.561684,-46.655981,-23.576519,-46.680050,Itaim Bibi,2,0,0,Gabriel,1,Julia,5.20.0,56,2.956139
1,DT01115,14,ramen,46.41,-23.561684,-46.655981,-23.537814,-46.573183,Tatuape,3,0,0,Carla,2,Otavio,5.20.0,71,8.847452
2,DT00061,13,feijoada,52.50,-23.561684,-46.655981,-23.543447,-46.661227,Perdizes,1,0,1,Marcos,2,Yasmin,6.0.0,50,2.097183
3,DT02173,12,ramen,38.13,-23.561684,-46.655981,-23.537306,-46.673415,Perdizes,1,1,1,Julia,2,Bruno,5.21.2,49,3.241316
4,DT04066,14,classic_burger,37.22,-23.561684,-46.655981,-23.540271,-46.576867,Tatuape,0,0,0,Bruno,2,Igor,5.20.0,50,8.408492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,DT02732,13,smash_burger,28.41,-23.561684,-46.655981,-23.558684,-46.683134,Pinheiros,0,1,1,Igor,1,Natalia,5.20.0,53,2.787626
3996,DT04110,12,ramen,41.58,-23.561684,-46.655981,-23.566982,-46.674104,Jardins,1,1,1,Igor,3,Elisa,5.20.0,59,1.938814
3997,DT03882,13,vegan_wrap,29.94,-23.561684,-46.655981,-23.597990,-46.712499,Morumbi,0,1,1,Elisa,2,Gabriela,5.18.0,54,7.033699
3998,DT02220,11,margherita_pizza,42.87,-23.561684,-46.655981,-23.568493,-46.687842,Pinheiros,0,0,1,Carla,2,Thiago,5.22.0,47,3.334436


In [ ]:
features = ["distance", "traffic_level" , "restaurant_busy_level", "order_price", "courier_vehicle",  "weather"]
target = ["delivery_time_minutes"]
x = deliveries_train[features]
y = deliveries_train[target]
x_train,x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [38]:
deliveries_train[features]

,distance,traffic_level,restaurant_busy_level,order_price,courier_vehicle,weather
0,2.956139,0,0,47.49,1,2
1,8.847452,0,0,46.41,2,3
2,2.097183,0,1,52.50,2,1
3,3.241316,1,1,38.13,2,1
4,8.408492,0,0,37.22,2,0
...,...,...,...,...,...,...
3995,2.787626,1,1,28.41,1,0
3996,1.938814,1,1,41.58,3,1
3997,7.033699,1,1,29.94,2,0
3998,3.334436,0,1,42.87,2,0


In [39]:
deliveries_train.corr(numeric_only=True)["delivery_time_minutes"].sort_values()

,delivery_time_minutes
customer_lat,-0.177765
customer_lng,-0.149226
courier_vehicle,0.062920
order_time,0.087325
weather,0.275897
order_price,0.346997
restaurant_busy_level,0.439103
traffic_level,0.441049
distance,0.578950
delivery_time_minutes,1.000000


In [ ]:

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
pipe.fit(x_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())])

In [ ]:
from math import sqrt
y_pred = pipe.predict(x_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

print(mae)

rmse = sqrt(mean_squared_error(
    y_test,
    y_pred,
))

print(rmse)

r2 = r2_score(y_test, y_pred)

print(r2)

5.309866925717408
6.7900320440404975
0.8090986144563533


In [ ]:
!pip install joblib

In [ ]:
import joblib

joblib.dump(pipe, "pipe.pkl")

['pipe.pkl']

In [ ]:
solution = pd.DataFrame({
    "delivery_time_minutes": y_pred.flatten(),
})

solution["delivery_time_minutes"] = solution["delivery_time_minutes"].round(0)
print(solution)

     delivery_time_minutes
0                     44.0
1                     57.0
2                     27.0
3                     33.0
4                     43.0
..                     ...
795                   42.0
796                   53.0
797                   60.0
798                   51.0
799                   34.0

[800 rows x 1 columns]


In [ ]:
deliveries_test = pd.read_csv("deliveries_test.csv")
deliveries_test["delivery_time_minutes"] = solution["delivery_time_minutes"]
deliveries_test.to_csv("deliveries_test.csv", index=False)